In [1]:
from qiskit import QuantumCircuit
import numpy as np

# 1. 定义问题规模和黑盒类型
n = 2 # 2 个输入比特
oracle_type = 'balanced' # 你可以把它改成 'constant' 试试看！

# 2. 初始化电路: n个输入位，1个辅助位，以及n个经典位用于测量
qc = QuantumCircuit(n+1, n)

# --- 步骤 1: 初始化辅助量子位为 |1> ---
qc.x(n)
qc.barrier()

# --- 步骤 2: 对所有量子位施加 H 门 ---
for qubit in range(n+1):
    qc.h(qubit)
qc.barrier()

# --- 步骤 3: 插入量子黑盒 (Oracle) ---
if oracle_type == 'constant':
    # 常数黑盒：f(x) = 0 或 1。这里我们随机决定。
    # 如果 f(x) = 0，什么都不做 (I 门)
    # 如果 f(x) = 1，对辅助位施加 X 门
    if np.random.randint(2) == 1:
        qc.x(n)

elif oracle_type == 'balanced':
    # 平衡黑盒：利用 CNOT 门。把每个输入位作为控制，辅助位作为目标。
    # 这确保了一半输入会让辅助位翻转，另一半不翻转。
    for qubit in range(n):
        qc.cx(qubit, n)
qc.barrier()

# --- 步骤 4: 再次对输入量子位施加 H 门 ---
for qubit in range(n):
    qc.h(qubit)
qc.barrier()

# --- 步骤 5: 测量输入量子位 ---
# 注意：我们不需要测量辅助量子位
for qubit in range(n):
    qc.measure(qubit, qubit)

print(f"当前测试的黑盒类型是: {oracle_type}")
print(qc.draw()) # 文本图

当前测试的黑盒类型是: balanced
           ░ ┌───┐ ░            ░ ┌───┐ ░ ┌─┐   
q_0: ──────░─┤ H ├─░───■────────░─┤ H ├─░─┤M├───
           ░ ├───┤ ░   │        ░ ├───┤ ░ └╥┘┌─┐
q_1: ──────░─┤ H ├─░───┼────■───░─┤ H ├─░──╫─┤M├
     ┌───┐ ░ ├───┤ ░ ┌─┴─┐┌─┴─┐ ░ └───┘ ░  ║ └╥┘
q_2: ┤ X ├─░─┤ H ├─░─┤ X ├┤ X ├─░───────░──╫──╫─
     └───┘ ░ └───┘ ░ └───┘└───┘ ░       ░  ║  ║ 
c: 2/══════════════════════════════════════╩══╩═
                                           0  1 
